# Task 2

## 2.1 Perform the linear regression on the provided dataset
### It's important that you divide your dataset in validation and train dataset (e.g.: in a 80-20 ratio),discuss and explain why.

The reason we are splitting the data in a training set and validation set is that we want to know how well our model performs. By using a validation set to test our model we can get an estimate on the "out-of-sample" error

### Discuss whether you should take a random sample in this case.

We should take a random sample since we want a model that can estimate the average income for a specific age. If we would train our model on the data sorted as is and on a 80-20 split, then the model would be trained on the ages 20-43, meaning that the model would be validated with data containing values it has never seen (ages 44-49). Since the aim is to model a linear model that best fits the data, interpolation (estimating values between known data points), and not extrapolation (estimating values outside the known data points) we need our validation set to be taken from within the same age range the model was trained on.


In [1]:
# Read the data
from reader_util import read_subset_file
# Prepare the data with a list and comprehension
cleaned_data = read_subset_file()
cleaned_data

[(20, 129.24285714285713),
 (21, 146.3095238095238),
 (22, 164.0),
 (23, 182.3952380952381),
 (24, 202.0761904761905),
 (25, 224.13809523809522),
 (26, 242.61904761904762),
 (27, 258.2333333333333),
 (28, 271.52857142857147),
 (29, 282.8333333333333),
 (30, 291.06666666666666),
 (31, 297.6047619047619),
 (32, 305.7095238095238),
 (33, 312.83809523809526),
 (34, 319.252380952381),
 (35, 327.87619047619046),
 (36, 335.7238095238095),
 (37, 343.14761904761906),
 (38, 354.1714285714286),
 (39, 362.8285714285714),
 (40, 371.0952380952381),
 (41, 379.6714285714286),
 (42, 386.147619047619),
 (43, 391.447619047619),
 (44, 400.34285714285716),
 (45, 405.5),
 (46, 408.8619047619048),
 (47, 414.06666666666666),
 (48, 415.0904761904762),
 (49, 416.2523809523809)]

## The linear regression
### Linear regression model prediction (vectorized form)
$$
y = h_\theta(x) = \theta \cdot x
$$
- $\theta$ is the model's parameter vector, containing the bias term $\theta_0$ and the feature weights $\theta_1$ to $\theta_n$.
- $x$ is the instance's feature vector, containing $x_0$ to $x_n$, with $x_0$ always equal to 1.
- $\theta \cdot x$ is the dot product of the vectors $\theta$ and $x$, which is of course equal to $\theta_0 x_0 + \theta_1 x_1 + \theta_2 x_2 + \cdots + \theta_n x_n$.
- $h_\theta$ is the hypothesis function, using the model parameters $\theta$.
### Normal equation:
To solve for the value of ${\theta}$ in the linear regression model we can use the normal equation
$$
\hat{\theta} = (X^T X)^{-1} X^T y
$$
- $\hat{\theta}$ is the value of  ${\theta}$ that minimizes the cost function.
- y is the vector of target values containing $y^1$ to $y^m$


In [ ]:
import numpy as np
import random
data_arr = np.array(cleaned_data)
rng = np.random.default_rng(42) # As given in Hands on Machine Learning 2nd Edition p.55 (Géron, 2019), 42 is "the answer to the ultimate question of life, the universe and everything" :)
rng.shuffle(data_arr)
train_set_size = int(len(data_arr) * 0.8)
train_set, test_set = data_arr[:train_set_size], data_arr[train_set_size:]

# Now we have the training set and test set.
# Now we create the feature matrix a 2D array with ones for the x0 and our age training set
# .T -> means transpose, and dot is the matrix multiplication
age_train = train_set[:,0]
income_train = train_set[:,1]
X_train = np.column_stack((np.ones(len(age_train)), age_train))
theta_hat = np.linalg.inv(X_train.T.dot(X_train)).dot(X_train.T).dot(income_train)



## 2.2 Create a scatter plot for your data and plot the regression line


In [ ]:
import matplotlib.pyplot as plt
age_data = data_arr[:,0]
income_data = data_arr[:,1]
X = np.column_stack((np.ones(len(age_data)), age_data))
sort_idx = np.argsort(age_data)
plt.scatter(age_data[sort_idx], income_data[sort_idx], color="blue", label="Training data")
y_predicted = theta_hat.dot(X.T) # We can also do X_train.dot(theta_hat), this is due to the different number of columns in the np arrays
plt.plot(age_data, y_predicted, color="red", label="Regression line")

In [ ]:
age_test = test_set[:,0]
income_test = test_set[:,1]
X_test = np.column_stack((np.ones(len(age_test)), age_test))
y_predicted_test = X_test.dot(theta_hat)
for real, pred in zip(income_test, y_predicted_test):
    print(f"Real: {real:.2f}, Predicted: {pred:.2f}")


### 2.4 Evaluate the model using MSE, implement the function that evaluates each of the real values against the predicted values (do not use sklearn MSE functions).

$$
MSE(X, h_\theta) = \frac{1}{m} \sum_{i=1}^{m} \left( \theta^T x^{(i)} - y^{(i)} \right)^2
$$


In [ ]:
mse = np.mean((income_test - y_predicted_test)**2)
print(f"MSE: {mse:.2f}")


### 2.5 Explain the obtained MSE value and what it means:

The mean squared error says that the model is usually off by $average\_income^2$ this does not say that much so we should take the square root so we have the same unit (average_income), the RMSE is 26.56. Meaning that the model usually misses by around 26.6 kSEK/year in average income. The linear regression is good for giving a rough estimate of the average income for an age. However, looking at the tails of the line, it is not accounting for younger and older ages, suggesting that a true relationship cannot be fully captured with a straight line.

In [ ]:
print(f"RMSE: {np.sqrt(mse):.2f}")